# Scratch Model With Qwen Tokenizer

This notebook trains this repo's scratch `TinyTransformer` using the local Qwen tokenizer from `models/qwen3-0.6b`.

This does not use Qwen weights. Only the tokenizer comes from Qwen.

## 1. Find The Repo Root And Use The Current Kernel

In [2]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

cwd = Path.cwd().resolve()
project_dir = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "step9_train_tiny_transformer.py").exists():
        project_dir = candidate
        break

if project_dir is None:
    raise RuntimeError("Could not find the repo root. Start VS Code from this project folder.")

os.chdir(project_dir)
print(f"project_dir: {project_dir}")
print(f"kernel python: {sys.executable}")

def run_py(script: str, *args: str) -> None:
    cmd = [sys.executable, "-u", script, *args]
    print(">", shlex.join(cmd))
    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)


project_dir: D:\project\llm
kernel python: d:\project\llm\.venv\Scripts\python.exe


## 2. Check CUDA And Local Qwen Tokenizer

In [3]:
import torch
from pathlib import Path

print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"cuda device: {torch.cuda.get_device_name(0)}")

qwen_dir = Path("models/qwen3-0.6b")
print(f"qwen tokenizer exists: {(qwen_dir / 'tokenizer.json').exists()}")
print(f"qwen config exists: {(qwen_dir / 'config.json').exists()}")


torch: 2.11.0+cu128
cuda available: True
cuda device: NVIDIA GeForce RTX 4080 Laptop GPU
qwen tokenizer exists: True
qwen config exists: True


## 3. Optional: Compare Tokenizers

In [3]:
# run_py(
#     "step26_compare_tokenizers.py",
#     "--text",
#     "Where is Hong Kong?",
#     "--limit",
#     "20",
# )

## 4. Prepare Data If Needed

Your full `data/fineweb_edu_sample_10bt.txt` is already present. Do not rerun `step29_prepare_fineweb_edu.py` unless you intentionally want to overwrite it. Build Qwen token shards once before full pretraining.

In [4]:
# FineWeb is already downloaded locally. Leave this commented to avoid overwriting it.
# run_py(
#     "step29_prepare_fineweb_edu.py",
#     "--max-chars",
#     "0",
# )

# Uncomment only when rebuilding Alpaca-cleaned SFT files.
# run_py("step30_prepare_alpaca_cleaned.py")

# Run once to tokenize the full FineWeb text into uint32 Qwen-token shards.
# This can take a while, but pretraining can resume from the saved shards after that.
#run_py("step32_tokenize_pretrain_shards.py")

## 5. Smoke Train With Qwen Tokenizer

This only checks that the Qwen tokenizer, data path, model shape, and CUDA training path work.

In [5]:
# Optional smoke test. Run this before a long pretrain if you changed tokenizer/model settings.
# run_py(
#     "step9_train_tiny_transformer.py",
#     "--preset",
#     "qwen_tokenizer_tiny_pretrain",
#     "--device",
#     "auto",
#     "--max-chars",
#     "100000",
#     "--max-iters",
#     "2",
#     "--save-every",
#     "0",
#     "--checkpoint",
#     "checkpoints/qwen_tokenizer_smoke/tiny_transformer.pt",
# )

## 6. Pretrain With Qwen Tokenizer Shards

Use `--resume` if you stop and continue later. This reads pretokenized memmap shards instead of loading the 46GB text file.

In [6]:
run_py(
    "step9_train_tiny_transformer.py",
    "--preset",
    "qwen_tokenizer_tiny_pretrain_sharded",
    "--device",
    "auto",
    "--resume",
    "--save-every",
    "10000",
)

> 'd:\project\llm\.venv\Scripts\python.exe' -u step9_train_tiny_transformer.py --preset qwen_tokenizer_tiny_pretrain_sharded --device auto --resume --save-every 10000
device: cuda
D:\project\llm\step9_train_tiny_transformer.py:585: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and amp_dtype == torch.float16))
amp: torch.bfloat16, grad_scaler: False
resumed checkpoint: checkpoints\qwen_tokenizer_tiny_pretrain\tiny_transformer.pt
resume step: 10001
torch.compile skipped (Triton not supported on Windows)
step 10020 / 50000 | loss 5.2775
step 10040 / 50000 | loss 5.3668
step 10060 / 50000 | loss 6.7155
step 10080 / 50000 | loss 5.4632
step 10100 / 50000 | loss 5.0023
step 10120 / 50000 | loss 5.5335
step 10140 / 50000 | loss 5.6633
step 10160 / 50000 | loss 5.5633
step 10180 / 50000 | loss 5.4422
step 10200 / 50000 | loss 5.5711
step 10220 / 50000 | loss 5.

KeyboardInterrupt: 

## 7. SFT With Qwen Tokenizer

In [ ]:
run_py(
    "step9_train_tiny_transformer.py",
    "--preset",
    "qwen_tokenizer_tiny_sft",
    "--device",
    "auto",
    "--resume",
    "--save-every",
    "1000",
)

## 8. Generate From Qwen-Tokenizer Scratch Model

In [ ]:
run_py(
    "step10_generate.py",
    "--preset",
    "qwen_tokenizer_tiny_sft",
    "--device",
    "auto",
    "--prompt",
    "How are you?",
    "--max-new-tokens",
    "80",
    "--temperature",
    "0.3",
)

In [ ]:
run_py(
    "step24_generate_qwen3.py",
    "--device",
    "auto",
    "--prompt",
    "Where is Hong Kong?",
)


In [12]:
run_py(
    "step10_generate.py",
    "--preset",
    "qwen_tokenizer_tiny_pretrain",
    "--device",
    "auto",
    "--prompt",
    "hello",
    "--max-new-tokens",
    "80",
    "--temperature",
    "0.4",
)

> 'd:\project\llm\.venv\Scripts\python.exe' -u step10_generate.py --preset qwen_tokenizer_tiny_pretrain --device auto --prompt hello --max-new-tokens 80 --temperature 0.4
device: cuda
hello of the same country. The main reason for this is to make sure that the country is not in the country. The only case of the country in the country is to be able to find the country of origin. The country is also an independent state of origin, but it is also the country of the country, the country of origin and the country. The country is the country of the country,
